In [2]:
# Imports

import ssl
import socket
import json
import urllib.parse
from datetime import datetime
import requests

In [3]:
url = "https://www.irib-news.ir/"

In [4]:
f"\n--- 1. Analyzing HTTP Headers for: {url} ---"

'\n--- 1. Analyzing HTTP Headers for: https://www.irib-news.ir/ ---'

In [5]:
# Send a standard GET request
response = requests.get(url, timeout=5, allow_redirects=True)
headers = response.headers

In [11]:
# Check for software version disclosure
disclosure_headers = ['Server', 'X-Powered-By', 'X-AspNet-Version']
found_info = False

# Let's see all the headers
for header, headerValue in headers.items():
    print(f"{header} => {headerValue}")

Server => Iransamaneh-1.7.0-s1
X-Cache => HIT
X-Cache-Hits => 925670
Content-Type => text/html; charset=utf-8
Expires => Mon, 26 Jul 1997 05:00:00 GMT
Cache-Control => post-check=0, pre-check=0
Pragma => no-cache
Date => Fri, 29 May 2026 02:11:02 GMT
Content-Length => 33683
Content-Encoding => gzip
Access-Control-Allow-Origin => *


In [12]:
for dh in disclosure_headers:
    if dh in headers:
        print(f"    - {dh}: {headers[dh]}")
        found_info = True
if not found_info:
    print("    - No obvious software versions disclosed in headers.")

    - Server: Iransamaneh-1.7.0-s1


So far, we know that the version of the server for this target is this : <br/>
`Iransamaneh-1.7.0-s1`

In [13]:
# Let's check the standard security headers
security_headers = [
    'Content-Security-Policy',
    'Strict-Transport-Security',
    'X-Content-Type-Options',
    'X-Frame-Options',
    'X-XSS-Protection'
]

for sh in security_headers:
    if sh in headers:
        print(f"    [✓] {sh}: {headers[sh][:50]}...")  # Truncated for readability
    else:
        print(f"    [✗] {sh} is MISSING")


    [✗] Content-Security-Policy is MISSING
    [✗] Strict-Transport-Security is MISSING
    [✗] X-Content-Type-Options is MISSING
    [✗] X-Frame-Options is MISSING
    [✗] X-XSS-Protection is MISSING


Ok, if I'm not mistaken, this means that this website doesn't have these security headers, right?

In [14]:
# And finally, let's see the `server` header

headers.get("Server", None)

'Iransamaneh-1.7.0-s1'

In [16]:
# Now we're done with software disclosures, let's check the SSL details

parsed_url = urllib.parse.urlparse(url)
target_hostname = parsed_url.hostname
port = 443

# Let's retrieve the SSL certificate details from the target hostname to evaluate validity and expiration.
f"\n--- 2. SSL/TLS Configuration Check for: {target_hostname} ---"


'\n--- 2. SSL/TLS Configuration Check for: www.irib-news.ir ---'

In [17]:
context = ssl.create_default_context()
with socket.create_connection((target_hostname, port), timeout=5) as sock:
    with context.wrap_socket(sock, server_hostname=target_hostname) as ssock:
        cert = ssock.getpeercert()
        cipher = ssock.cipher()

        print(f"** The established protocol: {cipher[1]}")
        print(f"** The cipher suite in use: {cipher[0]}")

        # Parse certificate validity dates
        not_after_str = cert.get('notAfter')
        not_after = datetime.strptime(not_after_str, '%b %d %H:%M:%S %Y %Z')
        remaining_days = (not_after - datetime.utcnow()).days

        print(f"** Certificate Expiration: {not_after} ({remaining_days} days remaining)")

        if remaining_days <= 0:
            print("    [!] WARNING: Certificate is EXPIRED!")
        elif remaining_days < 30:
            print("    [!] WARNING: Certificate expires soon (less than 30 days).")
        else:
            print("    [✓] Certificate validity is clear.")


** The established protocol: TLSv1.3
** The cipher suite in use: TLS_AES_256_GCM_SHA384
** Certificate Expiration: 2026-08-08 10:44:44 (71 days remaining)
    [✓] Certificate validity is clear.


C:\Users\hojat\AppData\Local\Temp\ipykernel_17576\1626076460.py:13: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  remaining_days = (not_after - datetime.utcnow()).days
